# 01_Data_Understanding

## Objective

The objective of this notebook is to understand the structure of the Olist E-Commerce dataset before performing any cleaning or analysis.

This notebook will:

- Load all relevant datasets
- Examine dataset dimensions
- Review column names and data types
- Identify missing values
- Understand relationships between tables

The findings from this notebook will guide subsequent data cleaning and business analysis steps.

In [2]:
import pandas as pd

# Display all columns when viewing dataframes
pd.set_option('display.max_columns', None)

## Step 1: Load Datasets

In [4]:
customers = pd.read_csv("../data/olist_customers_dataset.csv")
orders = pd.read_csv("../data/olist_orders_dataset.csv")
order_items = pd.read_csv("../data/olist_order_items_dataset.csv")
payments = pd.read_csv("../data/olist_order_payments_dataset.csv")
reviews = pd.read_csv("../data/olist_order_reviews_dataset.csv")
products = pd.read_csv("../data/olist_products_dataset.csv")
sellers = pd.read_csv("../data/olist_sellers_dataset.csv")

In [6]:
customers.head()

,customer_id,customer_unique_id,customer_zip_code_prefix,customer_city,customer_state
0,06b8999e2fba1a1fbc88172c00ba8bc7,861eff4711a542e4b93843c6dd7febb0,14409,franca,SP
1,18955e83d337fd6b2def6b18a428ac77,290c77bc529b7ac935b93aa66c333dc3,9790,sao bernardo do campo,SP
2,4e7b3e00288586ebd08712fdd0374a03,060e732b5b29e8181a18229c7b0b2b5e,1151,sao paulo,SP
3,b2b6027bc5c5109e529d4dc6358b12c3,259dac757896d24d7702b9acbbff3f3c,8775,mogi das cruzes,SP
4,4f2d8ab171c80ec8364f7c12e35b23ad,345ecd01c38d18a9036ed96c73b8d066,13056,campinas,SP


## Step 2: Understanding the size of each dataset

In [8]:

datasets = {
    "Customers": customers,
    "Orders": orders,
    "Order Items": order_items,
    "Payments": payments,
    "Reviews": reviews,
    "Products": products,
    "Sellers": sellers
}

for name, df in datasets.items():
    print(f"{name}: {df.shape}")

Customers: (99441, 5)
Orders: (99441, 8)
Order Items: (112650, 7)
Payments: (103886, 5)
Reviews: (99224, 7)
Products: (32951, 9)
Sellers: (3095, 4)


### First Observation

I noticed something interesting:

- Customers = 99,441
- Orders    = 99,441

At first glance, it almost looks like every customer placed exactly one order.

But as an analyst, I should never assume.

One of the key goals of this project is to analyze:

- Repeat purchases
- Customer loyalty
- Customer segments

So I need to verify whether customers can have multiple orders.

## Step 3: Examine the Columns

In [16]:
for name, df in datasets.items():
    print(f"\n{name}")
    print("-" * 50)
    print(df.columns.tolist())


Customers
--------------------------------------------------
['customer_id', 'customer_unique_id', 'customer_zip_code_prefix', 'customer_city', 'customer_state']

Orders
--------------------------------------------------
['order_id', 'customer_id', 'order_status', 'order_purchase_timestamp', 'order_approved_at', 'order_delivered_carrier_date', 'order_delivered_customer_date', 'order_estimated_delivery_date']

Order Items
--------------------------------------------------
['order_id', 'order_item_id', 'product_id', 'seller_id', 'shipping_limit_date', 'price', 'freight_value']

Payments
--------------------------------------------------
['order_id', 'payment_sequential', 'payment_type', 'payment_installments', 'payment_value']

Reviews
--------------------------------------------------
['review_id', 'order_id', 'review_score', 'review_comment_title', 'review_comment_message', 'review_creation_date', 'review_answer_timestamp']

Products
--------------------------------------------------


## Initial Data Model Assessment

Based on the dataset structure, the **Orders** table appears to be the central table in the Olist database. This table contains both the `order_id` and `customer_id` fields, suggesting that it serves as the primary link between customers and their purchasing activities.

### Potential Primary Keys

The following columns appear to uniquely identify records within their respective tables:

- **Customers:** `customer_id`
- **Orders:** `order_id`
- **Products:** `product_id`
- **Sellers:** `seller_id`
- **Reviews:** `review_id`

### Potential Relationships Between Tables

The initial inspection of the columns suggests the following relationships:

- `customer_id` links **Customers** and **Orders**
- `order_id` links **Orders** and **Order Items**
- `order_id` links **Orders** and **Payments**
- `order_id` links **Orders** and **Reviews**
- `product_id` links **Order Items** and **Products**
- `seller_id` links **Order Items** and **Sellers**

### Preliminary Database Structure

The dataset appears to follow a relational database design where:

- A customer places one or more orders.
- An order may contain one or more products.
- Products are supplied by sellers.
- Orders generate payment records.
- Customers may leave reviews for completed orders.

### Questions to Verify

Before performing any business analysis, the following assumptions must be validated using the data:

- Is `customer_id` unique within the Customers table?
- Is `order_id` unique within the Orders table?
- Can a customer place multiple orders?
- Can an order contain multiple products?
- Are there any duplicate records or missing relationships between tables?

These questions will be investigated in the next step to confirm the integrity and structure of the dataset.

## Step 4: Investigate

In [19]:
print("Unique customer_id in Customers:", customers["customer_id"].nunique())
print("Total rows in Customers:", len(customers))

print("\nUnique order_id in Orders:", orders["order_id"].nunique())
print("Total rows in Orders:", len(orders))

Unique customer_id in Customers: 99441
Total rows in Customers: 99441

Unique order_id in Orders: 99441
Total rows in Orders: 99441


## Validation of Primary Keys

To verify the integrity of the dataset, the uniqueness of the suspected primary key columns was assessed.

### Results

| Table | Key Column | Unique Values | Total Rows |
|---------|---------|---------:|---------:|
| Customers | customer_id | 99,441 | 99,441 |
| Orders | order_id | 99,441 | 99,441 |

### Findings

- Every record in the Customers table has a unique `customer_id`.
- Every record in the Orders table has a unique `order_id`.
- No duplicate customer or order identifiers were found.
- These results confirm that `customer_id` and `order_id` can be treated as primary keys for their respective tables.

### Observation

An interesting finding is that the Customers and Orders tables contain exactly the same number of records (99,441).

This raises an important analytical question:

> Does each customer place exactly one order, or does the dataset contain a different customer identifier structure?

This relationship will be investigated in the next step because it has significant implications for customer segmentation, retention analysis, and customer lifetime value calculations.

At first glance, this suggests that each customer placed exactly one order. However, the Customers table contains two different customer identifiers:

- `customer_id`
- `customer_unique_id`

This indicates that the dataset may distinguish between order-level customer records and actual unique customers.

### Next Step

The relationship between `customer_id` and `customer_unique_id` will be investigated to determine whether the same customer can appear multiple times in the dataset. This distinction is critical for future analyses such as:

- Customer Retention
- RFM Segmentation
- Repeat Purchase Analysis
- Customer Lifetime Value (CLV)

In [28]:
print("Unique customer_id:", customers["customer_id"].nunique())

print("Unique customer_unique_id:", customers["customer_unique_id"].nunique())

Unique customer_id: 99441
Unique customer_unique_id: 96096


## Customer Identifier Analysis

### Results

- Unique `customer_id`: 99,441
- Unique `customer_unique_id`: 96,096

### Findings

- The number of unique customers is lower than the number of customer records.
- This indicates that some customers made multiple purchases.
- `customer_unique_id` represents the actual customer and should be used for customer-level analyses.

### Key Insight

The dataset contains repeat customers, making it suitable for retention, segmentation, and customer lifetime value analyses.

In [37]:
customer_order_counts = (
    customers.groupby("customer_unique_id")
    .size()
    .reset_index(name="num_customer_ids")
)

customer_order_counts["num_customer_ids"].value_counts().sort_index().head(10)

num_customer_ids
1     93099
2      2745
3       203
4        30
5         8
6         6
7         3
9         1
17        1
Name: count, dtype: int64

## Repeat Customer Analysis

### Results

- 93,099 customers made 1 purchase.
- 2,745 customers made 2 purchases.
- A small number of customers made 3 or more purchases.
- The highest observed purchase count was 17.

### Findings

- Most customers purchased only once.
- A smaller group of customers returned and made multiple purchases.
- Repeat purchase behavior is present in the dataset.

### Key Insight

The customer base is dominated by one-time buyers, while a small segment of customers contributes multiple purchases. This makes customer retention and loyalty analysis particularly relevant for this project.

In [39]:
repeat_customers = (
    customer_order_counts["num_customer_ids"] > 1
).sum()

total_customers = len(customer_order_counts)

repeat_percentage = repeat_customers / total_customers * 100

print(f"Repeat Customers: {repeat_customers}")
print(f"Total Customers: {total_customers}")
print(f"Repeat Customer Rate: {repeat_percentage:.2f}%")

Repeat Customers: 2997
Total Customers: 96096
Repeat Customer Rate: 3.12%


## Repeat Customer Rate

### Results

- Total Customers: 96,096
- Repeat Customers: 2,997
- Repeat Customer Rate: 3.12%

### Findings

- Only a small proportion of customers made more than one purchase.
- Most customers placed a single order and did not return during the observed period.

### Key Insight

The dataset shows a low repeat customer rate (3.12%), suggesting that customer retention may be a challenge. This highlights the importance of future analyses such as customer segmentation, retention analysis, and customer lifetime value estimation.

In [42]:
orders.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 99441 entries, 0 to 99440
Data columns (total 8 columns):
 #   Column                         Non-Null Count  Dtype 
---  ------                         --------------  ----- 
 0   order_id                       99441 non-null  object
 1   customer_id                    99441 non-null  object
 2   order_status                   99441 non-null  object
 3   order_purchase_timestamp       99441 non-null  object
 4   order_approved_at              99281 non-null  object
 5   order_delivered_carrier_date   97658 non-null  object
 6   order_delivered_customer_date  96476 non-null  object
 7   order_estimated_delivery_date  99441 non-null  object
dtypes: object(8)
memory usage: 6.1+ MB


## Orders Table Overview

### Results

- The Orders table contains 99,441 records and 8 columns.
- All columns are currently stored as object data types.
- Several delivery-related columns contain missing values.

### Findings

- Date columns will need to be converted to datetime format for time-based analysis.
- Missing values are primarily found in approval and delivery timestamps.
- These missing values may be related to order status rather than data quality issues.

### Key Insight

Before handling missing values, the distribution of order statuses should be examined to determine whether the missing timestamps are expected for cancelled or unavailable orders.

In [45]:
orders["order_status"].value_counts()

order_status
delivered      96478
shipped         1107
canceled         625
unavailable      609
invoiced         314
processing       301
created            5
approved           2
Name: count, dtype: int64

In [48]:
delivered_percentage = (
    orders["order_status"].eq("delivered").mean() * 100
)

print(f"Delivered Order Rate: {delivered_percentage:.2f}%")

Delivered Order Rate: 97.02%


## Order Status Analysis & Delivered Order Rate

### Results

- Delivered Order Rate: 97.02%

### Findings

- The vast majority of orders were successfully delivered.
- Only a small percentage of orders were canceled, unavailable, or still in progress.

### Key Insight

Since delivered orders represent completed purchases, future customer and revenue analyses will focus primarily on delivered orders to ensure that metrics reflect actual customer transactions.




## Step 5: Data Preparation

In [53]:
# Convert the order date columns from text (object) to proper datetime format.

date_columns = [
    "order_purchase_timestamp",
    "order_approved_at",
    "order_delivered_carrier_date",
    "order_delivered_customer_date",
    "order_estimated_delivery_date"
]

for col in date_columns:
    orders[col] = pd.to_datetime(orders[col])

orders[date_columns].dtypes

order_purchase_timestamp         datetime64[ns]
order_approved_at                datetime64[ns]
order_delivered_carrier_date     datetime64[ns]
order_delivered_customer_date    datetime64[ns]
order_estimated_delivery_date    datetime64[ns]
dtype: object

## Date Conversion

### Results

- Converted all order-related date columns to datetime format:
  - order_purchase_timestamp
  - order_approved_at
  - order_delivered_carrier_date
  - order_delivered_customer_date
  - order_estimated_delivery_date

### Findings

- All date columns were successfully converted from object to datetime format.
- The dataset is now ready for time-based analyses and calculations.

### Key Insight

Converting date columns to datetime format enables future analyses such as sales trends, customer retention, cohort analysis, and delivery performance evaluation.

In [56]:
summary = pd.DataFrame({
    "Table": list(datasets.keys()),
    "Rows": [df.shape[0] for df in datasets.values()],
    "Columns": [df.shape[1] for df in datasets.values()]
})

summary

,Table,Rows,Columns
0,Customers,99441,5
1,Orders,99441,8
2,Order Items,112650,7
3,Payments,103886,5
4,Reviews,99224,7
5,Products,32951,9
6,Sellers,3095,4


## Dataset Summary

### Results

| Table | Rows | Columns |
|---------|---------:|---------:|
| Customers | 99,441 | 5 |
| Orders | 99,441 | 8 |
| Order Items | 112,650 | 7 |
| Payments | 103,886 | 5 |
| Reviews | 99,224 | 7 |
| Products | 32,951 | 9 |
| Sellers | 3,095 | 4 |

### Findings

- The dataset contains information on customers, orders, products, sellers, payments, and reviews.
- The Orders table serves as the central table connecting most business entities.
- The Order Items table contains more records than the Orders table, indicating that some orders contain multiple products.
- The dataset contains sufficient information for customer, product, revenue, retention, and seller analyses.

### Key Insight

The Olist dataset provides a comprehensive view of the e-commerce purchasing process, making it suitable for customer behavior analysis, retention analysis, and business performance evaluation.

# Conclusion

In this notebook, the structure and quality of the Olist e-commerce dataset were examined.

Key findings include:

- The dataset consists of seven core tables containing customer, order, product, payment, review, and seller information.
- Primary keys and table relationships were identified.
- Repeat customers were confirmed through the use of `customer_unique_id`.
- The repeat customer rate was found to be 3.12%.
- Approximately 97% of all orders were successfully delivered.
- Date columns were converted to datetime format to support future time-based analyses.

The dataset appears to be well-structured and suitable for customer analytics. The next step is to perform exploratory data analysis (EDA) to better understand customer purchasing behavior, order trends, and business performance.